# Coaching Agent — Session Demo

Simulates a full rehab session with two exercises.

**Timing rules:**
- Event arrives every **5s** (simulated with `time.sleep`)
- **30s** silence → exercise ends → exercise feedback generated
- **120s** silence → phase ends → phase report generated + JSON exported

**Session layout:**
```
0s        Exercise 1: Mini Squat  (6 events × 5s = 30s of activity)
30s       30s silence → Exercise 1 summary
60s       Exercise 2: Leg Press   (6 events × 5s = 30s of activity)
90s       120s silence → Phase report + JSON export
```

⚠️ Total runtime ≈ **5–6 minutes** (includes LLM generation time)

## 1. Setup

In [1]:
import sys, time
from pathlib import Path

# coaching_agent/ imports
sys.path.insert(0, str(Path('.').resolve()))

from session_manager import SessionManager

print('SessionManager loaded ✓')

SessionManager loaded ✓


## 2. Patient Profile

In [2]:
patient_profile = {
    "patient_id":        "P001",
    "condition":         "knee osteoarthritis",
    "condition_category":"knee",
    "rehab_phase":       "mid",
    "pain_level":        4,
    "weeks_into_rehab":  10,
    "age":               58,
    "goals":             "Walk dog 30 minutes daily without pain",
}

## 3. Define Simulated Events

Exercise 1: **Mini Squat** — knee valgus issue (high severity early, improving)

Exercise 2: **Leg Press** — forward lean (medium severity, stable)

In [3]:
def make_event(exercise_name, mistake_type, quality, severity,
               occurrences, rom_level, session_time):
    """Helper to build a coaching_event payload."""
    return {
        "coaching_event": {
            "exercise":   {"name": exercise_name, "confidence": 0.88},
            "mistake":    {
                "type":             mistake_type,
                "confidence":       0.75,
                "duration_seconds": 3.5,
                "persistence_rate": 0.4,
                "occurrences":      occurrences,
            },
            "metrics": {
                "speed_rps":        0.9,
                "rom_level":        rom_level,
                "height_level":     3,
                "torso_rotation":   0,
                "direction":        "none",
                "no_obvious_issue_p": 0.1,
            },
            "quality_score":        quality,
            "severity":             severity,
            "is_recoaching":        False,
            "session_time_minutes": session_time,
            "tier":                 "tier_2",
            "cache_key":            None,
            "routing_reason":       "form issue detected",
        },
        "session_id":       "session_P001_mid",
        "coaching_history": [],
    }


# Exercise 1: Mini Squat — quality improving over 6 events
events_ex1 = [
    make_event("mini squat", "knee valgus", 0.28, "high",   31, 1, 0.0),
    make_event("mini squat", "knee valgus", 0.31, "high",   28, 1, 0.1),
    make_event("mini squat", "knee valgus", 0.35, "medium", 22, 2, 0.2),
    make_event("mini squat", "knee valgus", 0.40, "medium", 18, 2, 0.3),
    make_event("mini squat", "knee valgus", 0.44, "medium", 15, 2, 0.4),
    make_event("mini squat", "knee valgus", 0.50, "low",    10, 3, 0.5),
]

# Exercise 2: Leg Press — forward lean, stable quality
events_ex2 = [
    make_event("leg press", "forward lean", 0.55, "medium", 20, 2, 2.0),
    make_event("leg press", "forward lean", 0.52, "medium", 18, 2, 2.1),
    make_event("leg press", "forward lean", 0.58, "medium", 15, 3, 2.2),
    make_event("leg press", "forward lean", 0.56, "low",    12, 3, 2.3),
    make_event("leg press", "forward lean", 0.60, "low",    10, 3, 2.4),
    make_event("leg press", "forward lean", 0.62, "low",     8, 3, 2.5),
]

print(f'Exercise 1 events: {len(events_ex1)}')
print(f'Exercise 2 events: {len(events_ex2)}')

Exercise 1 events: 6
Exercise 2 events: 6


## 4. Run Session

Feeds events every 5s. Background thread auto-triggers summaries and report.

**Expected output sequence:**
1. 6 × Exercise 1 event logs
2. 30s silence → `🏋️ EXERCISE ENDED: Mini Squat` + LLM feedback
3. 6 × Exercise 2 event logs  
4. 120s silence → `📋 REHAB PHASE ENDED` + LLM report + JSON path

In [4]:
sm = SessionManager(
    patient_profile=patient_profile,
    ollama_model="gemma3:4b",
    verbose=True,
)
sm.start()

Loading embeddings: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing vector DB from: E:\GRADUATE\Semster\26 Spr Semester\DS5500\agent\coaching_agent\chroma_coaching_db
Loaded 33782 document chunks
RAG knowledge base loaded ✓

  SESSION STARTED
  Patient : knee osteoarthritis
  Phase   : mid
  Pain    : 4/10



In [8]:
# ── Exercise 1 ────────────────────────────────────────────────────
print("--- Sending Exercise 1 events ---")
for payload in events_ex1:
    sm.ingest(payload)
    time.sleep(5)

print("\n[Waiting 35s for exercise timeout + LLM generation...]")
time.sleep(35)

# 主线程打印 Exercise 1 feedback
if sm.exercise_feedbacks:
    f = sm.exercise_feedbacks[-1]
    print(f"\n{'─'*60}")
    print(f"  🏋️  EXERCISE ENDED: {f['exercise_name']}")
    print(f"  Avg quality: {f['avg_quality']:.2f} | Trend: {f['quality_trend']}")
    print(f"{'─'*60}")
    print("\n  ❌ WHAT TO CORRECT:")
    for m in f["mistakes"]:
        print(f"    • {m['type']} ({m['severity']}) — {m['occurrences']} occurrences")
    print("\n  ✅ WHAT YOU DID WELL:")
    for ok in f["ok_aspects"]:
        print(f"    • {ok}")
    print(f"\n  💡 COACHING CUE:\n  {f['feedback']}\n")

# ── Exercise 2 ────────────────────────────────────────────────────
print("--- Sending Exercise 2 events ---")
for payload in events_ex2:
    sm.ingest(payload)
    time.sleep(5)

print("\n[Waiting 35s for exercise 2 timeout + LLM generation...]")
time.sleep(35)

# 主线程打印 Exercise 2 feedback
if len(sm.exercise_feedbacks) >= 2:
    f = sm.exercise_feedbacks[-1]
    print(f"\n{'─'*60}")
    print(f"  🏋️  EXERCISE ENDED: {f['exercise_name']}")
    print(f"  Avg quality: {f['avg_quality']:.2f} | Trend: {f['quality_trend']}")
    print(f"{'─'*60}")
    print("\n  ❌ WHAT TO CORRECT:")
    for m in f["mistakes"]:
        print(f"    • {m['type']} ({m['severity']}) — {m['occurrences']} occurrences")
    print("\n  ✅ WHAT YOU DID WELL:")
    for ok in f["ok_aspects"]:
        print(f"    • {ok}")
    print(f"\n  💡 COACHING CUE:\n  {f['feedback']}\n")

# ── Phase timeout ─────────────────────────────────────────────────
print("[Waiting 120s for phase timeout + report generation...]")
time.sleep(125)

print("Session complete ✓")

--- Sending Exercise 1 events ---
  ↳ event: mini squat | quality=0.28 | mistake=knee valgus
  ↳ event: mini squat | quality=0.31 | mistake=knee valgus
  ↳ event: mini squat | quality=0.35 | mistake=knee valgus
  ↳ event: mini squat | quality=0.40 | mistake=knee valgus
  ↳ event: mini squat | quality=0.44 | mistake=knee valgus
  ↳ event: mini squat | quality=0.50 | mistake=knee valgus

[Waiting 35s for exercise timeout + LLM generation...]

────────────────────────────────────────────────────────────
  🏋️  EXERCISE ENDED: Mini Squat
  Avg quality: 0.48 | Trend: improving
────────────────────────────────────────────────────────────

  ❌ WHAT TO CORRECT:
    • knee valgus (low) — 124 occurrences
    • forward lean (low) — 83 occurrences

  ✅ WHAT YOU DID WELL:
    • Good movement speed maintained throughout
    • Consistent squat depth achieved
    • Torso rotation well controlled

  💡 COACHING CUE:
  ✅ WHAT YOU DID WELL:
[You maintained good movement speed and consistently achieved a go

## 5. Inspect JSON Output

In [6]:
import json
from pathlib import Path

outputs_dir = Path('..') / 'phase_outputs'
json_files  = sorted(outputs_dir.glob('*.json'))

if not json_files:
    print('No JSON files found yet — phase may still be generating.')
else:
    latest = json_files[-1]
    print(f'Latest output: {latest.name}\n')
    with open(latest, encoding="utf-8") as f:
        data = json.load(f)

    print(json.dumps(data, indent=2))

Latest output: P001_mid_20260322_163523.json

{
  "phase_summary": {
    "patient_id": "P001",
    "condition": "knee osteoarthritis",
    "rehab_phase": "mid",
    "pain_level": 4,
    "weeks_into_rehab": 10,
    "goals": "Walk dog 30 minutes daily without pain",
    "phase_start_ts": 1774197165.48,
    "phase_end_ts": 1774197315.97,
    "phase_duration_s": 150.5
  },
  "exercises": [
    {
      "exercise_name": "Mini Squat",
      "event_count": 12,
      "quality_scores": [
        0.28,
        0.31,
        0.35,
        0.4,
        0.44,
        0.5,
        0.55,
        0.52,
        0.58,
        0.56,
        0.6,
        0.62
      ],
      "avg_quality": 0.48,
      "quality_trend": "improving",
      "mistakes": [
        {
          "type": "knee valgus",
          "occurrences": 124,
          "avg_duration_s": 3.5,
          "avg_persistence": 0.4,
          "severity": "low"
        },
        {
          "type": "forward lean",
          "occurrences": 83,
         

## 6. Quick JSON Summary

In [7]:
if json_files:
    ps = data['phase_summary']
    print(f"Patient    : {ps['patient_id']}")
    print(f"Condition  : {ps['condition']}")
    print(f"Phase      : {ps['rehab_phase']}")
    print(f"Duration   : {ps['phase_duration_s']:.0f}s")
    print(f"Pain level : {ps['pain_level']}/10")

    print(f"\nExercises completed: {len(data['exercises'])}")
    for ex in data['exercises']:
        print(f"  • {ex['exercise_name']}: "
              f"avg quality {ex['avg_quality']:.2f}, "
              f"trend {ex['quality_trend']}")
        for m in ex['mistakes']:
            print(f"      ↳ {m['type']} ({m['severity']}) — "
                  f"{m['occurrences']} occurrences")

    print(f"\nNext phase focus:")
    for f_ in data['next_phase_focus']:
        print(f"  → {f_}")

    print(f"\nOverall quality trend : {data['overall_quality_trend']}")

Patient    : P001
Condition  : knee osteoarthritis
Phase      : mid
Duration   : 150s
Pain level : 4/10

Exercises completed: 1
  • Mini Squat: avg quality 0.48, trend improving
      ↳ knee valgus (low) — 124 occurrences
      ↳ forward lean (low) — 83 occurrences

Next phase focus:
  → Correct knee valgus in next phase
  → Correct forward lean in next phase

Overall quality trend : improving



────────────────────────────────────────────────────────────
  🏋️  EXERCISE ENDED: Mini Squat
  Events: 12 | Avg quality: 0.48 | Trend: improving
────────────────────────────────────────────────────────────
  Generating exercise feedback...

  ✅ WHAT YOU DID WELL:
  [You maintained good movement speed and consistently achieved a good squat depth – fantastic!]
  ❌ WHAT TO CORRECT:
  [Reduce knee valgus – it puts extra stress on your knee. Focus on pushing your knees out slightly, like you're balancing a glass on them.]
  💡 CUE FOR NEXT SET:
  [Imagine pushing your knees out slightly during the mini squat – keep them aligned.]


  📋 REHAB PHASE ENDED
  Exercises completed: 1
  Generating phase report...

  ────────────────────────────────────────────────────────
  **End-of-Session Report – Patient: Knee Osteoarthritis – Week 10, Phase: Mid**

  **1. SESSION OVERVIEW:**
  Today’s session focused on reinforcing the Mini Squat exercise, continuing to build on the progress achieved over the